# Stage 2 — Element 검출 (YOLOv11-OBB, 회전 박스)

view 크롭(`data/view_crops`) 안에서 **공차/치수/GD&T/거칠기** 요소를 회전 박스(OBB)로 검출합니다.

**워크플로**
1. `data/view_crops/*.png` 를 **CVAT(rotated rectangle)** 로 라벨링
   - **효율화**: CVAT 라벨에 `value` **text attribute** 를 정의해 박스를 그릴 때 값을 같이 입력하면,
     한 번의 패스로 ⟨검출 라벨⟩+⟨Donut 값 라벨⟩ 두 데이터셋이 동시에 나옵니다.
     (값 라벨 추출은 `detection/element/cvat_to_donut.py` 참고)
2. CVAT → **"Ultralytics YOLO Oriented Bounding Boxes"** 포맷 export → `detection/element/cvat_export/`
3. split → 학습 → 검증 → 추론·정렬크롭

- 커널: **kardi_env** | 클래스: `element.yaml` (0=Dimension, 1=GD&T_FCF, 2=Datum, 3=Surface_Roughness, 4=Section, 5=Hole_Callout)

In [ ]:
# ── 환경 확인 ─────────────────────────────────────────────────────
from pathlib import Path
import ultralytics, torch
from ultralytics import YOLO
print("ultralytics", ultralytics.__version__, "| cuda", torch.cuda.is_available())
HERE = Path.cwd()
assert (HERE / "element.yaml").exists(), f"detection/element 에서 실행하세요 (현재: {HERE})"

In [ ]:
# ── CVAT OBB export → train/val 분리 (이미 배치된 데이터면 건너뜀) ──
import random, shutil
EXPORT_DIR = Path("cvat_export")            # images/, labels/ (8-좌표 OBB txt)
DST        = Path("datasets/element")
VAL_RATIO, SEED = 0.1, 42
src_img, src_lbl = EXPORT_DIR / "images", EXPORT_DIR / "labels"

if not src_img.exists():
    # cvat_export 가 없으면, 이미 datasets/element 에 배치된 데이터(예: raw_kaardi_data 이동본)를 사용.
    # 데이터를 덮어쓰지 않도록 split 을 건너뜁니다.
    pre = list((DST / "images" / "train").glob("*")) if (DST / "images" / "train").exists() else []
    assert pre, (f"{src_img} 도 없고 {DST}/images/train 도 비어 있습니다 — "
                 f"CVAT export 를 두거나 datasets/element 에 데이터를 배치하세요")
    n_val = len(list((DST / "images" / "val").glob("*"))) if (DST / "images" / "val").exists() else 0
    print(f"cvat_export 없음 → 배치된 데이터셋 사용 (train {len(pre)} / val {n_val}). split 건너뜀.")
else:
    stems = sorted(p.stem for p in src_img.glob("*.*")
                   if (src_lbl / (p.stem + ".txt")).exists())
    random.Random(SEED).shuffle(stems)
    n_val = max(1, int(len(stems) * VAL_RATIO))
    splits = {"val": stems[:n_val], "train": stems[n_val:]}
    for split, names in splits.items():
        for sub in ("images", "labels"):
            (DST / sub / split).mkdir(parents=True, exist_ok=True)
        for st in names:
            img = next(src_img.glob(st + ".*"))
            shutil.copy(img, DST / "images" / split / img.name)
            shutil.copy(src_lbl / (st + ".txt"), DST / "labels" / split / (st + ".txt"))
    print({k: len(v) for k, v in splits.items()})

In [ ]:
# ── 학습 (YOLOv11n-OBB) ──────────────────────────────────────────
# imgsz 는 view 크롭 크기에 맞춰 조정하세요. 회전 박스이므로 flip/회전 증강은 신중히.
model = YOLO("yolo11n-obb.pt")
model.train(
    data="element.yaml", epochs=200, imgsz=1024, batch=8,
    project="runs", name="element", exist_ok=True, seed=42,
    fliplr=0.0, flipud=0.0, degrees=0.0, mosaic=0.0,
    hsv_h=0.0, hsv_s=0.0, hsv_v=0.3, translate=0.05, scale=0.2,
)

In [ ]:
# ── 검증 (OBB mAP) ───────────────────────────────────────────────
best = YOLO("runs/element/weights/best.pt")
metrics = best.val(data="element.yaml")
print(f"mAP50 = {metrics.box.map50:.3f} | mAP50-95 = {metrics.box.map:.3f}")

In [ ]:
# ── 추론 → 정렬(rectify) element 크롭 저장 ───────────────────────
# 회전 박스를 수평으로 펴서 크롭 → Donut 입력 형태. (학습 데이터는 CVAT 라벨로 별도 생성)
import sys; sys.path.append("..")
from crop_utils import save_crops_from_result
best  = YOLO("runs/element/weights/best.pt")
CROPS = Path("../../../data/view_crops")
OUT   = Path("../../../data/elements_pred")     # 추론 산출(데모). 학습용은 cvat_to_donut.py 로 생성
n = 0
for img in sorted(CROPS.glob("*.png")):
    r = best.predict(img, imgsz=1024, conf=0.25, verbose=False)[0]
    n += len(save_crops_from_result(r, OUT, img.stem, pad=2))
print(f"정렬 element 크롭 {n}개 → {OUT}")

In [ ]:
# ── 정렬 크롭 육안 확인(글자가 수평으로 펴졌는지) ────────────────
import matplotlib.pyplot as plt
samples = sorted(Path("../../../data/elements_pred").glob("*.png"))[:8]
if samples:
    fig, axes = plt.subplots(1, len(samples), figsize=(3*len(samples), 3))
    axes = [axes] if len(samples) == 1 else axes
    for ax, p in zip(axes, samples):
        ax.imshow(plt.imread(p)); ax.axis("off"); ax.set_title(p.stem.split('__')[-1], fontsize=8)
    plt.show()
else:
    print("아직 크롭 없음 — 위 추론 셀을 먼저 실행하세요")